DataConfig
Setting for which dataset to load and also how to split it into train, val and test dataset.

In [6]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

import yaml


@dataclass
class DataConfig:
    dataset: str = "md17"          # "md17" | "md22"
    molecule: str = "ethanol"
    theory: str = "dft"            # "dft" | "ccsd" | "ccsd_t" — never pool across levels
    gold_path: str = ""            # training reads gold only, never bronze/silver directly
    cutoff_radius: float = 5.0     # Angstrom, for neighbor graph construction
    train_split: float = 0.8       # contiguous trajectory blocks, not random — see glossary.md §5
    val_split: float = 0.1
    test_split: float = 0.1
    use_literature_split: bool = False  # true for aspirin/malonaldehyde/toluene/ccsd_t-ethanol
    seed: int = 42

    def __post_init__(self) -> None:
        # Guard against silently-mismatched splits (e.g. a typo like 0.8/0.1/0.2).
        total = self.train_split + self.val_split + self.test_split
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"train/val/test splits must sum to 1.0, got {total}")

ModelConfig
Architecture choice and hyperparameters includes specific settings only used by certain phases.

In [7]:
@dataclass
class ModelConfig:
    phase: str = "mpnn"            # "mpnn" | "blip" | "graph_stochastic"
    hidden_channels: int = 128
    num_layers: int = 4
    num_rbf: int = 50              # Gaussian RBF basis size for edge distance expansion
    num_mc_samples: int = 20       # stochastic forward passes for UQ (blip / graph_stochastic only)
    perturbation_strength: float = 0.1  # graph_stochastic only

    def __post_init__(self) -> None:
        # Fail fast on a typo'd phase name rather than silently falling
        # back to unexpected behavior downstream.
        valid_phases = ("mpnn", "blip", "graph_stochastic")
        if self.phase not in valid_phases:
            raise ValueError(f"model.phase must be one of {valid_phases}, got {self.phase!r}")

`TrainConfig`
Epochs, batchsize,learning rate and the energy loss weighting.

In [8]:
@dataclass
class TrainConfig:
    epochs: int = 200
    batch_size: int = 32
    lr: float = 5e-4
    energy_loss_weight: float = 1.0
    force_loss_weight: float = 100.0  # forces conventionally weighted much higher than energy
    patience: int = 20
    device: str = "cpu"            # "cpu" | "cuda"

`ExperimentConfig`
Top level config that bundles the three above, loads from YAML and derives checkpoint results paths from the experiment name.

In [9]:
@dataclass
class ExperimentConfig:
    name: str = "unnamed_experiment"
    data: DataConfig = field(default_factory=DataConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    train: TrainConfig = field(default_factory=TrainConfig)

    @classmethod
    def from_yaml(cls, path: str | Path) -> "ExperimentConfig":
        # Build each sub-config from its own YAML section; missing
        # sections just fall back to that dataclass's defaults.
        path = Path(path)
        with open(path) as f:
            raw = yaml.safe_load(f)

        return cls(
            name=raw.get("name", path.stem),
            data=DataConfig(**raw.get("data", {})),
            model=ModelConfig(**raw.get("model", {})),
            train=TrainConfig(**raw.get("train", {})),
        )

    def checkpoint_path(self, checkpoints_dir: str | Path = "experiments/checkpoints") -> Path:
        # Checkpoint file is named after the experiment, so re-running
        # with the same name resumes/overwrites the same file.
        return Path(checkpoints_dir) / f"{self.name}.pt"

    def results_path(self, results_dir: str | Path = "experiments/results") -> Path:
        return Path(results_dir) / f"{self.name}.json"